In [ ]:
# env: conda activate pyucell
import scanpy as sc
import matplotlib.pyplot as plt
import pyucell as uc
import pandas as pd

In [ ]:
adata = sc.read_h5ad("c:/Users/irc/Desktop/Internship Bioinformatics 2025-2026/Lode/Annotation_Comparison/final_annotation_2026.h5ad")

In [ ]:
adata # X layer is log normalized already
# PCA space exists too, neighbours too, UMAP too, so do not calculate them again

In [ ]:
# this is a type mistake, which is still present in the adata object so we fix it manually here
adata.obs.loc[
    adata.obs["orig.ident"] == "SAM06",
    "treatment"
] = "WT"

In [ ]:
print(adata.obs["experiment"].unique().to_list())

pyUCell is a package for evaluating gene signatures in single-cell datasets. pyUCell signature scores, based on the Mann-Whitney U statistic, are robust to dataset size and heterogeneity, and their calculation demands less computing time and memory than other available methods, enabling the processing of large datasets in a few minutes even on machines with limited computing power.

Instead of making individual UMAP plots for each gene, pyUCell groups them to a module, and plots the score of that module on each cell in the dataset

A gene module is a group of genes that behave together and represent some underlying biological program

1) modules based on conserved markers of the celltypes across all experimental groups

In [ ]:
# for Late Mature first
# see "Downstream_processing.ipynb" for the conserved_markers for each celltype (calculation is somewhere in the middle, for bot WT vs Test and also across all experimental groups)
# but results are found at the end of the script
signatures = {
'Proliferating_cDC1': ['Ptma', 'Stmn1', 'Ran', 'Dut', 'Anp32b', 'Ranbp1', 'Hmgb1', 'Hnrnpab', 'Selenoh', 'Banf1', 'Snrpd1', 'Alyref', 'Dctpp1', 'Set', 'Lig1', 'Cdk4', 'Anp32e', 'Ssrp1', 'Hdgf', 'Sf3b5', 'Impdh2', 'Siva1', 'Pa2g4', 'Nudc', 'Dtymk', 'Smc1a', 'Hmgn1', 'Syce2', 'Nasp', 'AI506816', 'Cenpx', 'Lsm5', 'Rnaseh2c', 'Dpy30', 'Bex3', 'Ybx3', 'Fkbp4', 'Nop58', 'Snrpa1', 'Ddx39', 'Idh2', 'Fkbp3', 'Ndufa5', 'Lyar', 'Erh'],
'Late Immature': ['Dkk3', 'Itgae'],
'Early Mature': ['Cxcl9', 'Procr', 'Sdc4'],
'Late Mature': ['Epsti1', 'Tmem123', 'Cacnb3', 'Mreg', 'Il4i1', 'Traf1', 'Cd63', 'Tspan3', 'Relb', 'Myo1g', 'AW112010', 'Lamp1', 'Pcgf5', 'Fscn1', 'Adcy6', 'Rabgap1l', 'Glipr2', 'Psme2', 'Ankrd33b', 'Txndc17', 'Rnf115', 'Rogdi', 'Traf3', 'Socs2', 'Il21r', 'Serpinb9', 'Iscu', 'Lima1', 'Arhgef40', 'Birc2', 'Nudt17', 'Cers6', 'Lad1', 'Bcl2l14', 'Chka', 'Stk4', 'Ccser2', 'Laptm4b', 'Prex1', 'Arhgap28', 'Tnfrsf4', 'Adora2a', 'Adap1', 'Spsb1', 'Tmem19', 'Ric1', 'Nfat5', 'Rassf2', 'Map4', 'Tnfrsf11a', 'Anxa4', 'Dok1', 'Ogfrl1', 'Il2rg', 'Rassf3', 'Gtpbp1', 'Actn1', 'Tuba1a', 'Abcg1', 'Fas', 'Cpne2', 'Scpep1', 'Tpm1', 'Csrp1', 'Il15ra', 'Plcl2', 'Cd200', 'Ssh1', 'Atrx', 'Gucd1', 'Micu1', 'Ube2l6', 'Mif4gd', 'Slc22a23', 'Frmd4a', 'Polr3c', 'Lgmn', 'Iffo2', 'Net1', 'Kdm6a', 'Gbp8', 'Herc4', 'Atl3', 'Cmc2', 'Arc', 'Gbp4', 'Rap2b', 'Rrad', 'Slc26a2', 'Tmem150c', 'Tspo', 'Tmbim4', 'Slc2a6', 'Plekhm2', 'Phf21a', 'Synpo2', 'Gnb4', 'Ankib1', 'Fabp5', 'Pvr', 'Fbrs', 'Idh1', 'Map2k1', 'Uba7', 'Tubb2b', 'Ube2z', 'Clec2i', 'Triobp', 'Prr14', 'Pex13', 'Vwa5a', 'Hsf2', 'Myo1c', 'Gsdmd', 'Mmp25', 'Nfe2l1', 'Insl6', 'Fgfbp3', 'Sema7a', 'Krit1', 'Pdlim4', 'Marf1', 'Mex3b', 'Mtf1', 'Cdkn2b', 'Khnyn', 'D130040H23Rik', 'Runx2'],
'Other cDC1s': ['Gclc', 'Swap70', 'Ccr7', 'Il4i1', 'Mxd1', 'Traf1', 'Epsti1', 'Dusp5', 'Samsn1', 'Tspan3', 'Basp1', 'Rcsd1', 'Arhgap31', 'Etv3', 'Cacnb3', 'Il12b', 'Relb', 'Tmem123', 'Clic4', 'Klf6', 'Adam8', 'Fam49a', 'Serpinb9', 'Kdm2b', 'Chka', 'Plxnc1', 'Pik3r1', 'Gadd45b', 'Pfkfb3', 'Bmp2k', 'Ccl5', 'Cflar', 'Cdkn1a', 'Anxa3', 'Rgs1', 'Marcksl1', 'Myo1g', 'Cblb', 'Birc2', 'Fscn1', 'Arl5c', 'Ccdc88a', 'Selplg', 'Pcgf5', 'Cxcl16', 'Nampt', 'Akap13', 'Lrrk1', 'Zc3h12c', 'Rnf19b', 'Nabp1', 'Map4k4', 'Tmem176a', 'Rogdi', 'Dennd4a', 'AW112010', 'Zfp36l1', 'Bhlhe40', 'Tbc1d4', 'Wnk1', 'Cd200', 'Rel', 'Irf1', 'Adcy6', 'Csrp1', 'Ankrd33b', 'Nlrc5', 'Cd63', 'Atxn1', 'Uvrag', '4930523C07Rik', 'Tmem39a', 'Ncoa7', 'Rassf2', 'Gpr132', 'Apol7c', 'Ddhd1', 'Mllt6', 'Tmcc3', 'Ppp4r2', 'Rassf3', 'Nr4a3', 'Bcl2l14', 'Lima1', 'Socs2', 'Clec2d', 'Ccl22', 'Il4ra', 'Zfc3h1', 'Rab21', 'Slc6a6', 'Fchsd2', 'Nfkb2', 'Il21r', 'Aebp2', 'Poglut1', 'Ifrd1', 'Atp2b1', 'Grk3', 'Map3k14', 'Icosl', 'Spred1', 'Tle3', 'Gm38062', 'BC005537', 'Arhgap22', 'Spint2', 'Birc3', 'Ly75', 'Rnf115', 'A630081D01Rik', 'Tuba1a', 'Hmgcr', 'Map4', 'Gpr137b-ps', 'Sqstm1', 'Tob2', 'Gtf2a1', 'Ptger4', 'Tbc1d8', 'Mbp', 'Mreg', 'Actn1', 'Laptm4b', 'Pvr', 'Nfkbia', 'Mycbp2', 'Glipr2', 'Rftn1', 'Tmem176b', 'Nudt9', 'Cd40', 'Arhgef40', 'Txndc17', 'Bcl2l11', 'Specc1', 'Arl4c', 'Tnfaip3', 'Tnfrsf4', 'Etnk1', 'Kcnk6', 'Casp3', 'Gbp4', 'Col27a1', 'Gm44694', 'Lrrc8c', 'Dleu2', 'Strip2', 'Rab8b', 'Ric1', 'Malt1', 'Ilrun', 'Lamp1', 'Ccser2', 'Prex1', 'Zmynd15', 'Kmt2e', 'Frmd4a', 'Spsb1', 'Ehbp1l1', 'Kpna3', 'Inf2', 'Adora2a', 'Tpm1', 'Sinhcaf', 'Tnfrsf1b', 'Dok1', 'Ogfrl1', 'Cdk12', 'Cep350', 'Hivep1', 'Tbc1d1', 'Galnt7', 'Rasa4', 'Rasa2', 'Atxn7l1', 'Lactb', 'Sgpl1', 'H2-Eb2', 'Ccrl2', 'Vcam1', 'Stxbp3', 'Mical3', 'Phip', 'Rab11fip1', 'Ptafr', 'Atf7ip', 'Gucd1', 'Rrad', 'Cd274', 'Mapre2', 'Lpp', 'Tgif1', 'Clec7a', 'Eno3', 'Gm44292', 'Net1', 'Hmgcs1', 'Pnpla8', 'Prr14', 'Kmt2a', 'Atxn2l', 'Pdlim5', 'Gramd3', 'N4bp2l1', 'Sfmbt1', 'Adap1', 'Mab21l3', 'Specc1l', 'H2-M2', 'Oga', 'Ccng2', 'Plekhb2', 'Lncpint', 'Map3k8', 'Wdr91', 'Prnp', 'Snn', 'Rap2a', 'Tmem168', 'Nectin1', 'Abcg1', 'Slco5a1', 'Foxp4', 'Ankrd44', 'Zc3h12d', 'Trim8', 'Herc4', 'Src', 'Sel1l', 'Il2rg', 'Msi2', 'Arf2', 'Dip2b', 'Lad1', 'Rb1cc1', 'Rgs3', 'Gm43761', 'Reep3', 'Gca', 'Fbxl3', 'Dctn1', 'Ncoa2', 'Tank', '5330406M23Rik', 'Fnip2', 'Mau2', 'Aftph', 'Pakap.2', 'Cd80', 'Ahctf1', 'Zbtb10', 'Thap12', 'Gm34455', 'Gypc', 'Nudt17', 'Zmym5'],
'cDC1_engulfing_RBC': ['Hba-a1', 'Hba-a2', 'Hbb-bs']

# Early Immature and Pre_cDC1 have None 
}

In [ ]:
uc.compute_ucell_scores(adata, signatures=signatures)

In [ ]:
adata.obs

In [ ]:
sc.pl.umap(adata,
            color=['Proliferating_cDC1_UCell', 
                   'Late Immature_UCell', 
                   'Early Mature_UCell', 
                   'Late Mature_UCell', 
                   'cDC1_engulfing_RBC_UCell', 
                   'Other cDC1s_UCell'],
              cmap="viridis", 
              ncols = 2,
              size=10)

In [ ]:
ucell_cols = [
    'Proliferating_cDC1_UCell',
    'Late Immature_UCell',
    'Early Mature_UCell',
    'Late Mature_UCell',
    'cDC1_engulfing_RBC_UCell',
    'Other cDC1s_UCell'
]

for score in ucell_cols:
    
    print(f"\n===== {score} =====")
    
    for exp in adata.obs["experiment"].unique():
        
        adata_sub = adata[adata.obs["experiment"] == exp].copy()
        
        sc.pl.umap(
            adata_sub,
            color=score,
            cmap="viridis",
            size=10,
            title=f"{exp} — {score}"
        )

In [ ]:
# export the results to pdf
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

ucell_cols = [
    'Proliferating_cDC1_UCell',
    'Late Immature_UCell',
    'Early Mature_UCell',
    'Late Mature_UCell',
    'cDC1_engulfing_RBC_UCell',
    'Other cDC1s_UCell'
]

experiments = adata.obs["experiment"].unique()

with PdfPages("UCell_conserved_markers.pdf") as pdf:

    for score in ucell_cols:

        fig, axes = plt.subplots(
            nrows=(len(experiments) + 2) // 3,
            ncols=3,
            figsize=(15, 10)
        )

        axes = axes.flatten()

        vmin = adata.obs[score].min()
        vmax = adata.obs[score].max()

        for ax, exp in zip(axes, experiments):

            adata_sub = adata[
                adata.obs["experiment"] == exp
            ].copy()

            sc.pl.umap(
                adata_sub,
                color=score,
                cmap="viridis",
                vmin=vmin,
                vmax=vmax,
                size=10,
                title=exp,
                ax=ax,
                show=False
            )

        for ax in axes[len(experiments):]:
            ax.axis("off")

        fig.suptitle(
            f"UCell score: {score}",
            fontsize=18,
            fontweight="bold",
            y=0.99
        )

        fig.tight_layout(rect=[0, 0, 1, 0.96])

        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

2) Now, we will perform the same, but for modules of specific experiments (Toxo and LNP, and WT) --> failed, because no such genes exist, that are conserved across all celltypes in an experiment

3) Same thing, but for TFs inferred from the transcriptomics data, so no DESeq2 results

In [ ]:
# for Late Mature first
# see "Downstream_processing.ipynb" for the conserved_markers for each celltype (calculation is somewhere in the middle, for bot WT vs Test and also across all experimental groups)
# but results are found at the end of the script
signatures = {
# 'Pre_cDC1': ,
# 'Proliferating_cDC1': ['Mbd2', 'Tfdp1', 'Tfap2c', 'Noto', 'Tfam', 'Snai2', 'Dnmt3b', 'Hcfc1', 'Myb', 'Neurod2', 'Skil', 'Tal1', 'Ttf1', 'Satb2', 'Atf1', 'Fgf2', 'E2f5', 'Lhx2', 'Nr1h3', 'Rara'],
# 'Early Immature': ,
'Late Immature': ["Abl1", "Ahr", "Arid3b", "Ascl1", "Atf1", "Atf2", "Atf5", "Atrx",
  "Bach2", "Batf", "Bcl11b", "Bcl3", "Bhlhe40", "Bmal2", "Brd4", "Btg2",
  "Cdx1", "Cdx2", "Cebpd", "Cebpe", "Creb1", "Creb5", "Crx", "Dlx2",
  "Dlx4", "Dmtf1", "Dmtf1l", "Dnmt1", "Dot1l", "E2f5", "Ebf1", "Egr4",
  "Elf1", "Elf3", "Elf4", "Elf5", "Elk4", "Ep300", "Erg", "Esr1",
  "Etv1", "Etv4", "Fosb", "Fosl2", "Foxa2", "Foxc1", "Foxc2", "Foxg1",
  "Foxh1", "Foxo3", "Foxp3", "Gabpa", "Gata3", "Gfi1", "Gtf3a", "Hdac1",
  "Hdac3", "Hdac7", "Hdac9", "Hes1", "Hey2", "Hhex", "Hivep1", "Hivep2",
  "Hlf", "Hlx", "Hmga1", "Hmga1b", "Hmga2", "Hmgb2", "Hnf1b", "Hoxa5",
  "Hoxd1", "Hsf2", "Htatip2", "Irf1", "Irf2", "Irf3", "Irf4", "Irf5",
  "Irf9", "Irx1", "Jun", "Junb", "Jund", "Kat2b", "Kdm2a", "Kdm5a",
  "Kdm5c", "Kdm5d", "Klf11", "Klf3", "Klf8", "Klf9", "Kmt2b", "Lhx2",
  "Lrrfip1", "Mafg", "Meis1", "Meox2", "Mlxip", "Mlxipl", "Mnt", "Mta3",
  "Mtf2", "Nanog", "Neurog2", "Nfatc3", "Nfatc4", "Nfe2", "Nfe2l2",
  "Nfe2l3", "Nfia", "Nfib", "Nfic", "Nfil3", "Nfkb2", "Nkrf", "Nkx2-1",
  "Npm1", "Nr0b1", "Nr0b2", "Nr1d1", "Nr2f1", "Nr2f2", "Nr3c1", "Nr4a2",
  "Nr4a3", "Nr5a2", "Nrf1", "Nrg1", "Pax2", "Pax5", "Pax8", "Pdx1",
  "Phf20", "Pitx2", "Pknox1", "Plagl1", "Plagl2", "Pml", "Pou2f2",
  "Pou3f1", "Ppara", "Ppard", "Ppargc1a", "Prdm1", "Prdm2", "Preb",
  "Prox1", "Pura", "Rcor2", "Relb", "Rest", "Rorc", "Rxrb", "Sall2",
  "Satb2", "Sim2", "Sin3a", "Sirt1", "Smad6", "Smad7", "Smarca4", "Sox17",
  "Sox9", "Sp2", "Sp4", "Srebf1", "Stat1", "Stat2", "Stat3", "Stat5a",
  "Stat5b", "Tbp", "Tbpl2", "Tbx15", "Tcf7l2", "Tead4", "Tef", "Tet1",
  "Tfap2b", "Tfap4", "Tfcp2", "Tfdp2", "Tfe3", "Thrb", "Trim28", "Ttf1",
  "Usf1", "Usf2", "Wt1", "Ybx3", "Zfhx3", "Zfp384", "Zfp458", "Zfp46",
  "Zfp64", "Zgpat", "Zic1"],
# 'Early Mature': ,
# 'Late Mature':['Tfam', 'Gtf3a', 'Fosb', 'Esr2', 'Snai2', 'Zfp64', 'Mef2c', 'Skil', 'Zfp729a', 'Ppargc1a', 'Rara', 'Zfp458', 'Nkx2-1', 'Htatip2', 'Sp3', 'Gata3', 'Kmt2a', 'Tbx5', 'Tbp', 'Neurod2'],
# 'Other cDC1s': ,
# 'cDC1_engulfing_RBC': 

# Early Immature and Pre_cDC1 have None 
}

In [ ]:
uc.compute_ucell_scores(adata, signatures=signatures)

In [ ]:
sc.pl.umap(adata,
            color=[
                # 'Proliferating_cDC1_UCell', 
                   'Late Immature_UCell', 
                #    'Early Mature_UCell', 
                  # 'Late Mature_UCell', 
                #    'cDC1_engulfing_RBC_UCell', 
                #    'Other cDC1s_UCell',
                'experiment',
                'treatment'
                   ],
              cmap="viridis", 
              ncols = 1,
              size = 10
              )

4) Now, we try to find unique genes for a celltype in a specific experiment, meaning the gene should only be significantly present, in only one celltype in one experiment

In [ ]:
# for Late Mature first
# see "Downstream_processing.ipynb" for the conserved_markers for each celltype (calculation is somewhere in the middle, for bot WT vs Test and also across all experimental groups)
# but results are found at the end of the script
signatures = {
'Late Mature':['Satb1', 'Aff3', 'Acsf2', 'Pik3r5', 'Gsto1', 'Cyfip1', 'Mt1', 'Zbtb20', 'Tes', 'Nfkb1', 'Tmem243', 'Emd', 'Stk17b', 'Alcam', 'Sbds', 'Pde4d', 'Tmem131', 'Fmnl1', 'Il2ra', 'Sgca']
 
}




In [ ]:
uc.compute_ucell_scores(adata, signatures=signatures)

In [ ]:
sc.pl.umap(adata,
            color=[
                'Late Mature_UCell', 
                'experiment',
                'celltype_final_2026'
                ],
              cmap="viridis", 
              ncols = 1,
              size = 5
              )